In [ ]:
import os
import shutil
from functools import partial
from pathlib import Path
import json
import pandas as pd

from urbanopt_des.uo_cli_wrapper import UOCliWrapper
from tools.docker_uo_cli_wrapper import DockerUOCliWrapper
from geojson_modelica_translator.modelica.modelica_runner import ModelicaRunner
from geojson_modelica_translator.modelica.GMT_Lib.DHC.DHC_5G_WH_GHX_HPTrio_VariableDist import (
    DHC5GWasteHeatGHXwithHPTrioVariableDist,
)
from geojson_modelica_translator.system_parameters.system_parameters import (
    SystemParameters,
)
from urbanopt_des.urbanopt_analysis import URBANoptAnalysis
from urbanopt_des.urbanopt_geojson import DESGeoJSON as URBANoptGeoJSON

# -- Execution mode -----------------------------------------------------------
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
DOCKER_IMAGE_TAG = "urbanopt-cli:1.2-ubuntu22"
# ---------------------------------------------------------------------------

if USE_DOCKER:
    UO = partial(DockerUOCliWrapper, image_tag=DOCKER_IMAGE_TAG)
else:
    UO = UOCliWrapper

# Autoreload dependencies while iterating on wrapper code.
%load_ext autoreload
%autoreload 2

In [ ]:
# Get the working directories
workdir = Path().resolve()
print(f"Working directory: {workdir}")

analysis_dir = workdir / "esbe_2026"
if not analysis_dir.exists():
    analysis_dir.mkdir()

template_data_dir = workdir / "data" / "templates"

print(f"template_data_dir: {template_data_dir}")
print(f"analysis_dir: {analysis_dir}")

# Get the number of usable cores for parallel processing by looking at the system (n-1)
num_usable_cores = max(1, (os.cpu_count() or 1) - 1)
print(f"Number of usable cores for parallel processing: {num_usable_cores}")

# update weather flag
update_weather_location = False
new_weather_epw = "USA_FL_MacDill.AFB.747880_TMY3"
new_weather_climate_zone = "1A"

# new_weather_epw = "Lecco_IT_TMY"
# new_weather_climate_zone = "4A"

### Activity 4: DES/TEN

In [ ]:
# This is the same setup pattern as prior activities, but in a new directory
activity_4_analysis_dir = analysis_dir / "activity_4"
if not activity_4_analysis_dir.exists():
    activity_4_analysis_dir.mkdir()

uo_coincident = UO(
    activity_4_analysis_dir, "coincident_pre", template_dir=template_data_dir
)

uo_coincident.create_example_coincident_project()
uo_coincident.create_scenarios("class_project_coincident.json")

# Copy over to the coincident directory for activity 4
uo_coincident = uo_coincident.update_project_files("coincident")
uo_coincident.set_number_parallel(num_usable_cores)  # run sims in parallel

# Copy over the weather files
uo_coincident.copy_over_weather()

# Change weather file in feature + mapper files when enabled
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )

# Fix issues -- copy over the updated Baseline.rb mapper file
shutil.copy2(
    template_data_dir / "mappers" / "Baseline.rb",
    uo_coincident.project_path / "mappers" / "Baseline.rb",
)
# Copy over the base workflow from the template. This one includes all the EEMs that are needed in the class project since
# the class_project_workflow.json inherits from base_workflow.osw. This is a workaround to avoid having to patch the workflow file to add in the missing EEM steps.
shutil.copy2(
    template_data_dir / "mappers" / "base_workflow.osw",
    uo_coincident.project_path / "mappers" / "base_workflow.osw",
)

In [ ]:
# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

# Post-process and visualize the baseline
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_feature("class_project_coincident.json")

In [ ]:
scenario_path = uo_coincident.project_path / "baseline_scenario.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

# save these as the relative paths because they will be run from within docker

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# Now create the 4G system
uo_coincident.des_params(scenario_path, feature_path, sys_param_path)

In [ ]:
# fix values in the sys_params.json file for the 4G system
# "central_cooling_plant_parameters": {
#   "heat_flow_nominal": 8000000,        // 8 MW, was 7999 W
#   "cooling_tower_fan_power_nominal": 50000,
#   "mass_chw_flow_nominal": 85,         // ~Σ building chw flows
#   "chiller_water_flow_minimum": 20,    // ~0.25 × nominal
#   "mass_cw_flow_nominal": 100,         // ~1.15 × chw nominal — this prevents the freeze
#   ...
# }

# create the 4g model
uo_coincident.des_create(sys_param_path, feature_path, "four_g")

#### Do not run this, takes VERY long

In [ ]:
if False:
    # Run the 4G system with the Modelica runner for now, even
    # though the URBANopt-DES package also has a run command.

    # Run the original 4G model from the uo CLI flow
    mr = ModelicaRunner()

    des_scenario_name = "four_g"
    des_analysis_4g_dir = activity_4_analysis_dir / des_scenario_name

    # Print the full arguments so we can verify what is being set
    print(f"{des_scenario_name}.Districts.district")
    print(f"file_to_load={des_analysis_4g_dir / 'package.mo'}")
    print(f"run_path={des_analysis_4g_dir}")

    print(f"Running model in the following directory: {des_analysis_4g_dir}")

    # Check whether results from a previous run already exist
    results_path = (
        des_analysis_4g_dir
        / f"{des_scenario_name}.Districts.DistrictEnergySystem_results"
    )
    if results_path.exists():
        print("Results already exist, skipping model run.")
    else:
        # Use the gmt-om-runner to run the model, not the UO DES cli
        _, results_path = mr.run_in_docker(
            "compile_and_run",
            f"{des_scenario_name}.Districts.DistrictEnergySystem",
            file_to_load=f"{des_analysis_4g_dir / 'package.mo'}",
            run_path=des_analysis_4g_dir,
            # march 1 -> march 14
            start_time=5097600,
            stop_time=6307200,
        )

    # running 4G model for 2 weeks takes ~

### 5G System (CLI Version)
Do not use this one, it does not work. The system parameters file is not set up correctly and the des_create function fails.

In [ ]:
# create the 5G model from uo cli workflow
## THIS DOES NOT WORK, DO NOT USE...
# sys_param_path_5g = uo_coincident.project_path / "sys_params_5g.json"
# uo_coincident.des_create(sys_param_path_5g, feature_path, "five_g")

### 5G System (Template/GMT Version)

This section keeps 4G creation in `uo_cli` but generates a 5G model using the GMT template class `DHC5GWasteHeatGHXwithHPTrioVariableDist`.

Inputs reused from earlier cells:
- `feature_path`: URBANopt feature file
- `uo_coincident.project_path / run / baseline_scenario`: baseline OpenStudio loads
- `activity_4_analysis_dir`: output root for generated Modelica packages

In [ ]:
# Build a 5G scenario from the GMT template using the same URBANopt baseline inputs
# ******* WARNING -- this will remove the target 5G project directory if it exists *******

# Use paths defined in earlier cells
new_geojson_file = feature_path
uo_analysis_baseline_results = uo_coincident.project_path / "run" / "baseline_scenario"
des_analysis_agg_dir = activity_4_analysis_dir

if not uo_analysis_baseline_results.exists():
    raise FileNotFoundError(
        f"Baseline scenario results not found: {uo_analysis_baseline_results}"
    )

# Important: use the repo root (workdir) as relative_path so modelica:// paths
# resolve to esbe_2026/... and not esbe/esbe_2026/...
relative_path_root = workdir

system_parameter_file = (
    des_analysis_agg_dir / f"{new_geojson_file.stem}_system_params_5g.json"
)
system_parameter = SystemParameters()

# Generate system parameter JSON from URBANopt baseline outputs
system_parameter.csv_to_sys_param(
    model_type="time_series",
    sys_param_filename=system_parameter_file,
    scenario_dir=uo_analysis_baseline_results,
    feature_file=new_geojson_file,
    district_type="5G",
    relative_path=relative_path_root,
    modelica_load_filename="modelica.mos",
)

# Build the 5G template package
model = DHC5GWasteHeatGHXwithHPTrioVariableDist(system_parameter)

des_scenario_name = "five_g"
des_analysis_baseline_dir = des_analysis_agg_dir / des_scenario_name
print(f"DES analysis dir for this scenario is {des_analysis_baseline_dir}")

if des_analysis_baseline_dir.exists():
    shutil.rmtree(des_analysis_baseline_dir)

model.build_from_template(des_analysis_agg_dir, des_scenario_name)

# Save the scenario name for downstream post-processing
with open(des_analysis_baseline_dir / "analysis_name.txt", "w") as f:
    f.write(des_scenario_name)

print("Created 5G model")

In [ ]:
# Even for 2 weeks this will take hours to run.
if False:
    # Run the generated 5G template model
    mr = ModelicaRunner()

    des_scenario_name = "five_g"
    des_analysis_baseline_dir = activity_4_analysis_dir / des_scenario_name

    if not (des_analysis_baseline_dir / "package.mo").exists():
        raise FileNotFoundError(
            f"Expected generated 5G package not found: {des_analysis_baseline_dir / 'package.mo'}"
        )

    # Print arguments for traceability
    print(f"{des_scenario_name}.Districts.district")
    print(f"file_to_load={des_analysis_baseline_dir / 'package.mo'}")
    print(f"run_path={des_analysis_baseline_dir}")
    print(f"Running model in the following directory: {des_analysis_baseline_dir}")

    # Skip run if result file already exists
    results_path = (
        des_analysis_baseline_dir / f"{des_scenario_name}.Districts.district_results"
    )
    if results_path.exists():
        print("Results already exist, skipping model run.")
    else:
        success, results_path = mr.run_in_docker(
            "compile_and_run",
            f"{des_scenario_name}.Districts.district",
            file_to_load=f"{des_analysis_baseline_dir / 'package.mo'}",
            run_path=des_analysis_baseline_dir,
            # march 1 -> march 14: 2 weeks takes XX minutes to run
            step_size=900,  # 15 min step size to speed up the run
            start_time=5097600,
            stop_time=6307200,
        )
        print(f"Run success: {success}")

    print(f"5G results path: {results_path}")

### 5G model that is an aggregated representation of the loads


In [ ]:
uo_analysis_baseline_dir = uo_coincident.project_path
uo_analysis_baseline_scenario_name = "baseline_scenario"
uo_analysis_baseline_results = (
    uo_analysis_baseline_dir / "run" / uo_analysis_baseline_scenario_name
)
project_geojson_filename = feature_path
des_analysis_agg_dir = activity_4_analysis_dir
relative_path_root = workdir

uo_analysis = URBANoptAnalysis(project_geojson_filename, des_analysis_agg_dir)

# Currently only one URBANopt result for now
uo_analysis.add_urbanopt_results(
    uo_analysis_baseline_dir, uo_analysis_baseline_scenario_name
)

# Process the building load measure reports
uo_analysis.urbanopt.process_load_results(uo_analysis.geojson.get_building_ids())

# save the URBANopt dataframes -- not really needed here
uo_analysis.urbanopt.save_dataframes()
uo_analysis.urbanopt.display_name = "Non-Connected"

# 1. Create an abstracted DES project which will be used to aggregate the loads.
# Output is written to: <uo_project_path>/run/baseline_scenario/all_buildings/
uo_analysis.urbanopt.create_abstract_run(
    "all_buildings", uo_analysis.urbanopt.data_loads
)

# 2. Copy geojson file
new_geojson_file = des_analysis_agg_dir / project_geojson_filename.name
shutil.copy(project_geojson_filename, des_analysis_agg_dir)

# 3. Read and update the GeoJSON file to aggregate all buildings
agg_geojson = URBANoptGeoJSON(new_geojson_file)
gdf_json = agg_geojson.create_aggregated_representation(["all"])

# Save the updated GeoJSON file
with open(new_geojson_file, "w") as json_file:
    json.dump(gdf_json, json_file, indent=2)

# 4. Copy and update the scenario file
new_scenario_file = des_analysis_agg_dir / scenario_path.name
shutil.copy(scenario_path, des_analysis_agg_dir)
scenario_df = pd.read_csv(new_scenario_file)
scenario_df = scenario_df.head(1)  # Keep only the first row for aggregation
scenario_df["Feature Id"] = "all_buildings"
scenario_df["Feature Name"] = "All Buildings"
scenario_df.to_csv(new_scenario_file, index=False)

# 5. Generate the system parameters file for the aggregated representation
system_parameter_file = (
    des_analysis_agg_dir / f"{new_geojson_file.stem}_system_params.json"
)
system_parameter = SystemParameters()
system_parameter.csv_to_sys_param(
    model_type="time_series",
    sys_param_filename=system_parameter_file,
    scenario_dir=uo_analysis_baseline_results,
    feature_file=new_geojson_file,
    district_type="5G",
    relative_path=relative_path_root,
    modelica_load_filename="modelica.mos",
)

# Build the 5G template package
model = DHC5GWasteHeatGHXwithHPTrioVariableDist(system_parameter)

des_scenario_name = "five_g_agg"
des_analysis_baseline_dir = des_analysis_agg_dir / des_scenario_name
if des_analysis_baseline_dir.exists():
    shutil.rmtree(des_analysis_baseline_dir)

model.build_from_template(des_analysis_agg_dir, des_scenario_name)

# Save the scenario name for downstream post-processing
with open(des_analysis_baseline_dir / "analysis_name.txt", "w") as f:
    f.write(des_scenario_name)

print("Created 5G aggregated model")

In [ ]:
# Run full year?
full_year_run = True

# Run the generated 5G aggregated template model
mr = ModelicaRunner()

des_scenario_name = "five_g_agg"
des_analysis_baseline_dir = activity_4_analysis_dir / des_scenario_name

if not (des_analysis_baseline_dir / "package.mo").exists():
    raise FileNotFoundError(
        f"Expected generated 5G package not found: {des_analysis_baseline_dir / 'package.mo'}"
    )

# Print arguments for traceability
print(f"{des_scenario_name}.Districts.district")
print(f"file_to_load={des_analysis_baseline_dir / 'package.mo'}")
print(f"run_path={des_analysis_baseline_dir}")
print(f"Running model in the following directory: {des_analysis_baseline_dir}")

# Skip run if result file already exists
results_path = (
    des_analysis_baseline_dir / f"{des_scenario_name}.Districts.district_results"
)
if results_path.exists():
    print("Results already exist, skipping model run.")
else:
    if full_year_run:
        success, results_path = mr.run_in_docker(
            "compile_and_run",
            f"{des_scenario_name}.Districts.district",
            file_to_load=f"{des_analysis_baseline_dir / 'package.mo'}",
            run_path=des_analysis_baseline_dir,
            step_size=300,  # 5 min step size to speed up the run
            stop_time=31536000,  # full year, takes 100 minutes to run ******
        )
    else:
        success, results_path = mr.run_in_docker(
            "compile_and_run",
            f"{des_scenario_name}.Districts.district",
            file_to_load=f"{des_analysis_baseline_dir / 'package.mo'}",
            run_path=des_analysis_baseline_dir,
            # march 1 -> march 14: 2 weeks takes 20 minutes to run
            step_size=300,  # 5 min step size to speed up the run
            start_time=5097600,
            stop_time=6307200,
        )
        print(f"Run success: {success}")

print(f"5G results path: {results_path}")